<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-02-model-architecture/lesson-2.2-transformer-from-scratch/notebooks/exercises-2_2.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2.2 — Code a Transformer from Scratch | Netsetos GenAI Course

> Build the exact architecture that powers GPT, Gemini, and Claude — piece by piece in PyTorch. By the end, you'll have a working Transformer you trained yourself.

---

## About this notebook

This notebook contains the **13 runnable code examples** from the published lesson `lesson-2.2.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Input Embeddings — Words to Vectors
2. Positional Encoding — Teaching Word Order
3. Self-Attention — The Core Breakthrough
4. Multi-Head Attention — Multiple Perspectives
5. Feed-Forward Network — The Thinking Layer
6. Layer Norm + Residual Connections — Training Stability
7. Encoder Block — Stack All Encoder Parts
8. Decoder Block — The Text Generator
9. Full Transformer — Assemble & Train!
10. Full Transformer — Assemble & Train!
11. The Modern "LLaMA Recipe" — What Changed Since 2017
12. KV-Cache — Why Inference Is Slow (and How to Fix It)
13. Parameter Counting — How Big Is Your Model?


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q torch numpy


## Lesson code

Each section below corresponds to one code block from the published lesson.


#### Input Embeddings — Words to Vectors

_Block 1 of 13_

STEP 1: INPUT EMBEDDINGS — Convert token IDs → dense vectors


In [ ]:
# =============================================
# STEP 1: INPUT EMBEDDINGS
# Convert token IDs → dense vectors
# Each word gets a 512-dimensional "GPS coordinate"
# in meaning space.
# =============================================

import torch
import torch.nn as nn
import math

class InputEmbedding(nn.Module):
    """
    Converts token IDs into dense vectors.
    
    Imagine a giant lookup table:
      Token 0 ('the')   → [0.12, -0.34, 0.56, ...]  (512 numbers)
      Token 1 ('cat')   → [0.78, 0.23, -0.11, ...]   (512 numbers)
      Token 2 ('sat')   → [-0.45, 0.67, 0.33, ...]   (512 numbers)
      ... (one row for every token in the vocabulary)
    
    These numbers are LEARNED during training.
    Initially random, they gradually become meaningful.
    """
    
    def __init__(self, vocab_size: int, d_model: int):
        # vocab_size = how many unique tokens exist (e.g., 32000)
        # d_model = size of each vector (e.g., 512)
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
    
    def forward(self, x):
        # x shape: (batch_size, sequence_length)
        # e.g., (32, 100) = 32 sentences, each 100 tokens long
        
        # Look up the vector for each token
        # Output shape: (batch_size, seq_len, d_model)
        # e.g., (32, 100, 512)
        return self.embedding(x) * math.sqrt(self.d_model)
        # ↑ The sqrt scaling is from the original paper.
        # It prevents the embeddings from being too small
        # compared to the positional encoding (Step 2).

# Quick test
emb = InputEmbedding(vocab_size=32000, d_model=512)
fake_tokens = torch.randint(0, 32000, (2, 10))  # 2 sentences, 10 tokens each
output = emb(fake_tokens)
print(f"Input shape:  {fake_tokens.shape}")   # (2, 10)
print(f"Output shape: {output.shape}")       # (2, 10, 512)
print(f"✅ Each token is now a 512-dim vector!")


#### Positional Encoding — Teaching Word Order

_Block 2 of 13_

STEP 2: POSITIONAL ENCODING — Add a unique "position fingerprint" to each


In [ ]:
# =============================================
# STEP 2: POSITIONAL ENCODING
# Add a unique "position fingerprint" to each
# word so the model knows word order.
#
# Uses sin/cos waves at different frequencies.
# Think of it like a clock: the second hand moves
# fast, the minute hand moves slower, the hour
# hand slowest. Combined, they tell you the
# EXACT time. Similarly, combined waves tell
# the model the EXACT position of each word.
# =============================================

class PositionalEncoding(nn.Module):
    
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create a matrix of shape (max_len, d_model)
        # Each row = the positional encoding for one position
        pe = torch.zeros(max_len, d_model)
        
        # Position indices: [0, 1, 2, ..., max_len-1]
        position = torch.arange(0, max_len).unsqueeze(1).float()
        
        # The "frequency" divisor — creates waves of different speeds
        # Low dimensions = fast waves (like seconds)
        # High dimensions = slow waves (like hours)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * 
            (-math.log(10000.0) / d_model)
        )
        
        # Even dimensions get sine, odd dimensions get cosine
        pe[:, 0::2] = torch.sin(position * div_term)  # dims 0,2,4,6...
        pe[:, 1::2] = torch.cos(position * div_term)  # dims 1,3,5,7...
        
        # Add batch dimension: (1, max_len, d_model)
        pe = pe.unsqueeze(0)
        
        # Register as buffer (saved with model, but not trained)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x shape: (batch, seq_len, d_model) — from Step 1
        # Add positional encoding to the embeddings
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# Quick test
pe = PositionalEncoding(d_model=512)
fake_emb = torch.randn(2, 10, 512)  # 2 sentences, 10 tokens, 512 dims
output = pe(fake_emb)
print(f"Input shape:  {fake_emb.shape}")  # (2, 10, 512)
print(f"Output shape: {output.shape}")   # (2, 10, 512) — same! Just added position info
print(f"✅ Each token now knows WHERE it is in the sentence!")


#### Self-Attention — The Core Breakthrough

_Block 3 of 13_

STEP 3: SCALED DOT-PRODUCT SELF-ATTENTION — The heart of the Transformer.


In [ ]:
# =============================================
# STEP 3: SCALED DOT-PRODUCT SELF-ATTENTION
# The heart of the Transformer.
#
# For each word:
#   1. Create Q, K, V by multiplying with learned matrices
#   2. Score = how much does my Q match each K?
#   3. Normalize scores to probabilities (softmax)
#   4. Output = weighted sum of V using those probabilities
# =============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    The actual attention formula from the paper:
    Attention(Q, K, V) = softmax(Q·K^T / √d_k) · V
    
    Step by step:
    1. Q·K^T → score matrix (how similar is each Q to each K?)
    2. / √d_k → scale down (prevents scores from being too large)
    3. softmax → convert to probabilities (0-1, sum to 1)
    4. · V → weighted combination of values
    """
    d_k = Q.size(-1)  # dimension of keys (e.g., 64)
    
    # Step 1: Score = Q · K^T
    # Shape: (batch, heads, seq_len, seq_len)
    # Each cell [i][j] = "how much should word i attend to word j?"
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Step 2: Scale by √d_k
    # Without this, scores get too large → softmax becomes spiky
    # → gradients vanish → training breaks
    scores = scores / math.sqrt(d_k)
    
    # Optional: apply mask (used in decoder to prevent "peeking ahead")
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # Step 3: Softmax → probabilities
    # Each row sums to 1.0
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 4: Weighted combination of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Quick test — let's see attention in action
batch, seq_len, d_k = 1, 4, 8  # 1 sentence, 4 words, 8-dim vectors
Q = torch.randn(batch, seq_len, d_k)
K = torch.randn(batch, seq_len, d_k)
V = torch.randn(batch, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)

print(f"Q shape: {Q.shape}")            # (1, 4, 8)
print(f"Output shape: {output.shape}")  # (1, 4, 8)
print(f"\n📊 Attention weights (who pays attention to whom):")
print(f"   Each row = one word. Values = how much it attends to each other word.")
print(weights.squeeze().detach().numpy().round(3))
# Each row sums to 1.0 ✅


#### Multi-Head Attention — Multiple Perspectives

_Block 4 of 13_

STEP 4: MULTI-HEAD ATTENTION — Run 8 attention operations in parallel,


In [ ]:
# =============================================
# STEP 4: MULTI-HEAD ATTENTION
# Run 8 attention operations in parallel,
# each looking for different patterns.
#
# Head 1 might learn: "who is the subject?"
# Head 2 might learn: "what is the action?"
# Head 3 might learn: "is there negation?"
# ... and so on
# =============================================

class MultiHeadAttention(nn.Module):
    
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model          # e.g., 512
        self.num_heads = num_heads      # e.g., 8
        self.d_k = d_model // num_heads # e.g., 64 (each head gets 64 dims)
        
        # Learned projection matrices
        # These transform the input into Q, K, V
        self.W_q = nn.Linear(d_model, d_model)  # Query projection
        self.W_k = nn.Linear(d_model, d_model)  # Key projection
        self.W_v = nn.Linear(d_model, d_model)  # Value projection
        self.W_o = nn.Linear(d_model, d_model)  # Output projection
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # 1. Project inputs to Q, K, V
        Q = self.W_q(query)  # (batch, seq_len, d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # 2. Split into multiple heads
        # Reshape: (batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)
        # Think of it as: 8 smaller attention operations running in parallel
        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Apply attention to each head
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # 4. Concatenate all heads back together
        # (batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model
        )
        
        # 5. Final linear projection
        output = self.W_o(attn_output)
        return self.dropout(output)

# Quick test
mha = MultiHeadAttention(d_model=512, num_heads=8)
x = torch.randn(2, 10, 512)  # 2 sentences, 10 tokens, 512 dims
out = mha(x, x, x)  # Self-attention: Q=K=V=same input
print(f"Input:  {x.shape}")   # (2, 10, 512)
print(f"Output: {out.shape}") # (2, 10, 512) — same shape, richer representation!
print(f"✅ 8 attention heads ran in parallel!")


#### Feed-Forward Network — The Thinking Layer

_Block 5 of 13_

STEP 5: FEED-FORWARD NETWORK — Two linear layers with ReLU in between.


In [ ]:
# =============================================
# STEP 5: FEED-FORWARD NETWORK
# Two linear layers with ReLU in between.
# Expands to 4x wider, then compresses back.
# Think: "expand to think, compress to summarize"
# =============================================

class FeedForward(nn.Module):
    
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        # d_model = 512, d_ff = 2048 (4x expansion)
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),   # 512 → 2048 (expand)
            nn.ReLU(),                   # Non-linearity
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),   # 2048 → 512 (compress back)
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)

# Quick test
ff = FeedForward(d_model=512, d_ff=2048)
x = torch.randn(2, 10, 512)
print(f"In: {x.shape} → Out: {ff(x).shape}")  # Same shape ✅


#### Layer Norm + Residual Connections — Training Stability

_Block 6 of 13_

STEP 6: RESIDUAL CONNECTION + LAYER NORM — The "glue" that makes deep Transformers trainable.


In [ ]:
# =============================================
# STEP 6: RESIDUAL CONNECTION + LAYER NORM
# The "glue" that makes deep Transformers trainable.
# =============================================

class ResidualConnection(nn.Module):
    """
    Pattern: LayerNorm → Sublayer → Dropout → Add original input
    
    The "Add" part is the residual connection:
      output = input + sublayer(LayerNorm(input))
    
    This means: even if the sublayer learns NOTHING,
    the input passes through unchanged. The sublayer
    only needs to learn the DIFFERENCE (residual).
    """
    
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer):
        # sublayer is a function (attention or feed-forward)
        return x + self.dropout(sublayer(self.norm(x)))
        #      ↑ residual: add original input back


#### Encoder Block — Stack All Encoder Parts

_Block 7 of 13_

STEP 7: ENCODER BLOCK — One layer = self-attention + feed-forward


In [ ]:
# =============================================
# STEP 7: ENCODER BLOCK
# One layer = self-attention + feed-forward
# both wrapped with residual + norm.
# Stack 6 of these = full encoder.
# =============================================

class EncoderBlock(nn.Module):
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)
    
    def forward(self, x, src_mask=None):
        # Sub-layer 1: Self-Attention
        x = self.residual1(x, lambda x: self.self_attn(x, x, x, src_mask))
        # Sub-layer 2: Feed-Forward
        x = self.residual2(x, self.feed_forward)
        return x

class Encoder(nn.Module):
    """Stack N encoder blocks."""
    
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

# Test: 6 encoder layers, just like the original paper
encoder = Encoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)
x = torch.randn(2, 10, 512)
print(f"In: {x.shape} → Out: {encoder(x).shape}")
print(f"✅ Full 6-layer encoder is working!")
print(f"   Parameters: {sum(p.numel() for p in encoder.parameters()):,}")


#### Decoder Block — The Text Generator

_Block 8 of 13_

STEP 8: DECODER BLOCK — Three sub-layers:


In [ ]:
# =============================================
# STEP 8: DECODER BLOCK
# Three sub-layers:
#   1. MASKED Self-Attention (can't peek ahead)
#   2. Cross-Attention (looks at encoder output)
#   3. Feed-Forward (processes the information)
# =============================================

class DecoderBlock(nn.Module):
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.masked_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)
        self.residual3 = ResidualConnection(d_model, dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # 1. Masked self-attention (decoder attends to itself)
        x = self.residual1(x, lambda x: self.masked_attn(x, x, x, tgt_mask))
        
        # 2. Cross-attention (decoder attends to encoder output)
        # Q = decoder, K & V = encoder
        # "What in the input is relevant to what I'm generating?"
        x = self.residual2(x, lambda x: self.cross_attn(x, encoder_output, encoder_output, src_mask))
        
        # 3. Feed-forward
        x = self.residual3(x, self.feed_forward)
        return x

class Decoder(nn.Module):
    """Stack N decoder blocks."""
    
    def __init__(self, d_model, num_heads, d_ff, num_layers, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        return self.norm(x)


#### Full Transformer — Assemble & Train!

_Block 9 of 13_

STEP 9: THE FULL TRANSFORMER 🏗️ — Everything assembled into one model.


In [ ]:
# =============================================
# STEP 9: THE FULL TRANSFORMER 🏗️
# Everything assembled into one model.
#
# Input text → Encoder → Decoder → Output text
# =============================================

class Transformer(nn.Module):
    
    def __init__(
        self,
        src_vocab_size: int,   # source language vocabulary
        tgt_vocab_size: int,   # target language vocabulary
        d_model: int = 512,    # embedding dimension
        num_heads: int = 8,    # attention heads
        num_layers: int = 6,   # encoder/decoder layers
        d_ff: int = 2048,      # feed-forward inner dimension
        max_len: int = 5000,   # max sequence length
        dropout: float = 0.1,
    ):
        super().__init__()
        
        # Embedding layers (Step 1)
        self.src_embedding = InputEmbedding(src_vocab_size, d_model)
        self.tgt_embedding = InputEmbedding(tgt_vocab_size, d_model)
        
        # Positional encoding (Step 2)
        self.positional_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        # Encoder (Steps 3-7)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        
        # Decoder (Step 8)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        
        # Final output projection
        # Converts d_model dimensions → vocabulary size (logits)
        self.output_projection = nn.Linear(d_model, tgt_vocab_size)
    
    def encode(self, src, src_mask=None):
        # Source text → embeddings → + position → encoder
        src = self.src_embedding(src)
        src = self.positional_encoding(src)
        return self.encoder(src, src_mask)
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        # Target text → embeddings → + position → decoder
        tgt = self.tgt_embedding(tgt)
        tgt = self.positional_encoding(tgt)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        # Full forward pass
        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
        logits = self.output_projection(decoder_output)
        return logits

# =============================================
# CREATE AND INSPECT THE MODEL
# =============================================

model = Transformer(
    src_vocab_size=32000,  # English vocabulary
    tgt_vocab_size=32000,  # Hindi vocabulary
    d_model=512,
    num_heads=8,
    num_layers=6,
    d_ff=2048,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"🏗️ TRANSFORMER MODEL CREATED!")
print(f"   Total parameters:     {total_params:>12,}")
print(f"   Trainable parameters: {trainable:>12,}")
print(f"   Model size (approx):  {total_params * 4 / 1e6:.1f} MB")

# Test with fake data
src = torch.randint(0, 32000, (2, 20))  # 2 English sentences, 20 tokens
tgt = torch.randint(0, 32000, (2, 25))  # 2 Hindi sentences, 25 tokens

output = model(src, tgt)
print(f"\n   Source shape:   {src.shape}")     # (2, 20)
print(f"   Target shape:   {tgt.shape}")     # (2, 25)
print(f"   Output shape:   {output.shape}")  # (2, 25, 32000)
print(f"   ↑ For each of 25 target positions, we get a probability")
print(f"     distribution over 32,000 possible next tokens.")
print(f"\n🎉 YOUR TRANSFORMER IS ALIVE!")


#### Full Transformer — Assemble & Train!

_Block 10 of 13_

TRAIN YOUR TRANSFORMER! — Task: reverse a sequence of numbers


In [ ]:
# =============================================
# TRAIN YOUR TRANSFORMER!
# Task: reverse a sequence of numbers
# Input:  [1, 2, 3, 4, 5]
# Output: [5, 4, 3, 2, 1]
# =============================================

import torch
import torch.nn as nn

# Tiny Transformer for this task
model = Transformer(
    src_vocab_size=20,    # numbers 0-19
    tgt_vocab_size=20,
    d_model=64,           # small for fast training
    num_heads=4,
    num_layers=2,
    d_ff=128,
    dropout=0.1,
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Create causal mask (decoder can't see future tokens)
def create_causal_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return ~mask  # True = can attend, False = blocked

# Generate training data: reverse sequences
def make_batch(batch_size=32, seq_len=8):
    src = torch.randint(1, 15, (batch_size, seq_len))
    tgt = src.flip(dims=[1])  # Reverse!
    return src, tgt

# Training loop
print("🏋️ Training Transformer to reverse sequences...\n")

for epoch in range(200):
    src, tgt = make_batch()
    
    # Teacher forcing: feed actual target (shifted right) as input
    tgt_input = tgt[:, :-1]   # All tokens except last
    tgt_label = tgt[:, 1:]    # All tokens except first
    
    tgt_mask = create_causal_mask(tgt_input.size(1))
    
    # Forward pass
    output = model(src, tgt_input, tgt_mask=tgt_mask)
    
    # Compute loss
    loss = criterion(
        output.reshape(-1, output.size(-1)),
        tgt_label.reshape(-1)
    )
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1:3d} | Loss: {loss.item():.4f}")

# Test it!
print(f"\n🧪 TESTING (does it reverse correctly?):\n")
model.eval()
with torch.no_grad():
    test_src, test_tgt = make_batch(batch_size=5, seq_len=6)
    tgt_mask = create_causal_mask(test_tgt[:, :-1].size(1))
    pred = model(test_src, test_tgt[:, :-1], tgt_mask=tgt_mask)
    pred_tokens = pred.argmax(dim=-1)
    
    for i in range(5):
        src_list = test_src[i].tolist()
        expected = test_tgt[i][1:].tolist()
        predicted = pred_tokens[i].tolist()
        match = "✅" if expected == predicted else "❌"
        print(f"  Input:    {src_list}")
        print(f"  Expected: {expected}")
        print(f"  Got:      {predicted} {match}\n")

print("🎉 Your hand-built Transformer learned to reverse sequences!")


### The Modern "LLaMA Recipe" — What Changed Since 2017

Every production LLM uses 4 upgrades. Here's what they are and how to implement them.

_Block 11 of 13_

STEP 10: The Modern "LLaMA Recipe" — 4 Key Upgrades


In [ ]:
# =====================================================
# STEP 10: The Modern "LLaMA Recipe" — 4 Key Upgrades
# =====================================================
import torch
import torch.nn as nn
import math

# ── Upgrade 1: RMSNorm replaces LayerNorm ──
# 77% of modern models use this. Removes mean-centering,
# keeping only the scaling — faster, equally effective.
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight
        # vs LayerNorm: no mean subtraction, just scale by RMS

# ── Upgrade 2: SwiGLU replaces ReLU ──
# 72% of modern models. Gated activation: 3 weight matrices
# but hidden dim reduced from 4d to 8d/3 to match params.
class SwiGLUFFN(nn.Module):
    def __init__(self, d_model, hidden=None):
        super().__init__()
        hidden = hidden or int(d_model * 8 / 3)  # Match param count
        self.w1 = nn.Linear(d_model, hidden, bias=False)
        self.w3 = nn.Linear(d_model, hidden, bias=False)  # Gate
        self.w2 = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        return self.w2(nn.functional.silu(self.w1(x)) * self.w3(x))
        # SiLU gate * linear → project back. Better than ReLU!

# ── Upgrade 3: RoPE replaces sinusoidal positional encoding ──
# 70% of models. Rotates Q,K vectors by position-dependent angles.
# Encodes RELATIVE position via rotation, not absolute position.
def apply_rope(x, freqs):
    """Apply Rotary Position Embedding to Q or K."""
    d = x.shape[-1]
    x_pairs = x.view(*x.shape[:-1], d // 2, 2)
    x_complex = torch.view_as_complex(x_pairs.float())
    freqs = freqs[:x.shape[-2], :]
    rotated = x_complex * freqs
    return torch.view_as_real(rotated).flatten(-2).type_as(x)

def precompute_rope_freqs(d_head, max_seq=2048, theta=10000.0):
    """Precompute rotation frequencies for RoPE."""
    freqs = 1.0 / (theta ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq)
    angles = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(angles), angles)

# ── Upgrade 4: Grouped Query Attention (GQA) ──
# Shares K,V heads across multiple Q heads.
# LLaMA 3 70B: 64 Q heads, 8 KV heads → 8x KV-cache savings!
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads  # How many Q heads per KV

        self.wq = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(n_heads * self.head_dim, d_model, bias=False)

    def forward(self, x, freqs, mask=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(q, freqs)
        k = apply_rope(k, freqs)

        # Repeat K,V heads to match Q head count
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out)

print("""
THE LLAMA RECIPE — 4 upgrades from 2017 → 2025:
  1. LayerNorm    → RMSNorm    (faster, no mean centering)
  2. ReLU/GELU    → SwiGLU     (gated activation, 3 matrices)
  3. Sinusoidal   → RoPE       (rotary position, relative encoding)
  4. Full MHA     → GQA        (shared K,V heads, 8x cache savings)
  + Pre-norm instead of post-norm
  + No bias terms in linear layers
""")


### KV-Cache — Why Inference Is Slow (and How to Fix It)

The #1 inference bottleneck. Without KV-cache, generating 100 tokens requires recomputing attention for ALL previous tokens at EACH step.

_Block 12 of 13_

STEP 11: KV-Cache — The Inference Bottleneck


In [ ]:
# =====================================================
# STEP 11: KV-Cache — The Inference Bottleneck
# =====================================================
import torch
import torch.nn as nn
import math

class CachedAttention(nn.Module):
    """Attention with KV-Cache for efficient autoregressive generation."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

    def forward(self, x, kv_cache=None):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # The KEY insight: append new K,V to cache, don't recompute!
        if kv_cache is not None:
            prev_k, prev_v = kv_cache
            k = torch.cat([prev_k, k], dim=2)  # Append new K
            v = torch.cat([prev_v, v], dim=2)  # Append new V

        new_cache = (k, v)  # Store for next token

        # Attention: Q attends to ALL K,V (cached + new)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.wo(out), new_cache

# Autoregressive generation with KV-cache
def generate_with_cache(model, prompt_tokens, max_new=50):
    """
    Two-phase generation:
      1. PREFILL: Process entire prompt at once (compute-bound)
      2. DECODE:  Generate one token at a time using cache (memory-bound)
    """
    kv_cache = None

    # Phase 1: PREFILL — process full prompt, build cache
    out, kv_cache = model(prompt_tokens, kv_cache=kv_cache)
    next_token = out[:, -1:, :].argmax(dim=-1)  # Last token prediction

    # Phase 2: DECODE — one token at a time, reuse cache
    generated = [next_token]
    for _ in range(max_new):
        # Only process the NEW token, not the full sequence!
        out, kv_cache = model(next_token, kv_cache=kv_cache)
        next_token = out[:, -1:, :].argmax(dim=-1)
        generated.append(next_token)

    return generated

print("""
KV-CACHE transforms inference from O(n²) to O(n):
  Without cache: generating token 100 recomputes attention for ALL 100 tokens
  With cache:    generating token 100 only computes Q for token 100,
                 reuses K,V from tokens 1-99 stored in cache

Memory impact (LLaMA 3 70B, 32K context, batch=8):
  Full MHA cache:  ~100 GB (FP16)
  With GQA (8 KV heads): ~12.5 GB  ← 8x savings!

Two phases:
  PREFILL  = process prompt    → compute-bound (big matrix multiply)
  DECODE   = generate tokens   → memory-bound (read all weights for 1 token)
  
  This is why long prompts process fast but generation feels slower.
""")


### Parameter Counting — How Big Is Your Model?

Every GenAI engineer should be able to estimate model size, memory requirements, and cost from a config file.

_Block 13 of 13_

STEP 12: Parameter Counting & Memory Estimation


In [ ]:
# =====================================================
# STEP 12: Parameter Counting & Memory Estimation
# =====================================================

def count_params(d_model, n_heads, n_layers, vocab_size, n_kv_heads=None):
    """Count parameters for a decoder-only Transformer."""
    n_kv_heads = n_kv_heads or n_heads  # Full MHA if not specified
    head_dim = d_model // n_heads
    ffn_hidden = int(d_model * 8 / 3)  # SwiGLU standard

    # Per layer
    attn_q = d_model * (n_heads * head_dim)         # Q projection
    attn_k = d_model * (n_kv_heads * head_dim)      # K projection (GQA)
    attn_v = d_model * (n_kv_heads * head_dim)      # V projection (GQA)
    attn_o = (n_heads * head_dim) * d_model          # Output projection
    ffn = d_model * ffn_hidden * 3                   # SwiGLU: 3 matrices
    norm = d_model * 2                               # 2 RMSNorm per layer
    per_layer = attn_q + attn_k + attn_v + attn_o + ffn + norm

    # Global
    embedding = vocab_size * d_model
    final_norm = d_model
    lm_head = d_model * vocab_size  # Often tied with embedding

    total = n_layers * per_layer + embedding + final_norm + lm_head
    return total, per_layer, embedding

# Real model configs
models = [
    ("GPT-2",           768,  12, 12, 50257,  12),
    ("LLaMA 3 8B",      4096, 32, 32, 128256, 8),
    ("LLaMA 3 70B",     8192, 64, 80, 128256, 8),
    ("Our Transformer", 512,  8,  6,  10000,  8),
]

print(f"{'Model':<20} {'Params':>12} {'FP16 GB':>10} {'INT4 GB':>10}")
print("-" * 55)
for name, d, nh, nl, vs, nkv in models:
    total, _, _ = count_params(d, nh, nl, vs, nkv)
    fp16_gb = total * 2 / 1e9   # 2 bytes per param
    int4_gb = total * 0.5 / 1e9 # 0.5 bytes per param
    print(f"  {name:<18} {total/1e9:>10.2f}B {fp16_gb:>9.1f} {int4_gb:>9.1f}")

print("""
MEMORY RULES OF THUMB:
  FP16: params × 2 bytes  (7B model ≈ 14 GB)
  INT4: params × 0.5 bytes (7B model ≈ 3.5 GB → fits 1 GPU!)
  
  + KV-cache: (2 × n_layers × n_kv_heads × head_dim × seq_len × batch) × 2 bytes
  
  LLaMA 3 8B at 4K context, batch=1:
    Model:    16 GB (FP16)
    KV-cache:  0.5 GB
    Total:    ~17 GB → fits A100 40GB with room for batching

  LLaMA 3 70B at 32K context, batch=8:  
    Model:    140 GB → needs 4× A100 80GB
    KV-cache: ~100 GB (full MHA) or ~12 GB (GQA)
    That's why GQA is essential for production!
""")


---

## Wrap-up

You've walked through all 13 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
